# Parking V0 development baseline local first
**PARKING-V0-LOCAL-FIRST-2026-10-29-r2**

Use a **CPU runtime**. No CUDA setup or GPU is required. The model, weights, 512 development images, preprocessing and metrics are unchanged. We have changed **file handling only**.

**New flow:** one dev-only input ZIP → local disk → local evaluation and logs → one archived Drive checkpoint every four completed data shards → one final result ZIP. Held-out image/GT shards are never packaged or opened.

The first input package must be created while Drive is readable. It reads only the frozen manifest, selected V0 bundle and 20 dev ZIPs (about 76 MB of image/GT shards), not the 45 GB source dataset or old coverage chunks. Once packaged, future sessions copy one archive. An active Drive quota cannot be reset by this notebook. Stop repeated attempts if Colab reports a quota.

You can alternatively upload the prepared capsule through Colab's Files panel and set USE_DRIVE=False. That runs without Drive reads or writes; download your results before the VM is deleted. Never make dataset files public to work around a quota.

This is depth on GT-labelled surfaces, not detection mAP or ultrasonic nearest-surface accuracy. No per-image scale fitting, ±10 cm gate, V1 training or phone update.

## 1 Set paths and update main
Use a new local-first RUN_ID, preserving the failed r1 run. Keep the same RUN_ID for resuming this workflow. Stop the previous process before restarting. A different CPU/package runtime still requires a new RUN_ID because predictions must not be mixed.

For offline input, set USE_DRIVE=False and upload the already prepared parking_v0_dev_inputs.zip to /content using the Files panel. On a fresh VM, restore the latest successfully saved portable checkpoint; work since that save may repeat. If Drive saving failed, download a local progress archive before leaving. In the same VM, every completed local prediction shard is reusable.

In [ ]:
from pathlib import Path
import collections, datetime, json, os, shlex, subprocess, sys
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
REVISION = 'PARKING-V0-LOCAL-FIRST-2026-10-29-r2'
RUN_ID = 'parking_v0_actual_camera_epoch4_local_r2'
if not RUN_ID or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_ID): raise ValueError('Invalid RUN_ID')
BASE = Path('/content/drive/MyDrive')
MOBILE = Path('/content/mobile_adas3d')
BENCHMARK_SOURCE = BASE/'mobile_adas3d_outputs/parking_v0_v1_benchmark/parking_v0_v1_frozen_benchmark_r1'
MODEL_SOURCE = BASE/'mobile_adas3d_outputs/parking_shared_adapter/shared_adapter_frozen_fullpool_r1/parking_shared_adapter_results.zip'
REMOTE_CAPSULE = BASE/'mobile_adas3d_inputs/parking_v0_dev/epoch4_frozen512_r1/parking_v0_dev_inputs.zip'
REMOTE_CHECKPOINTS = BASE/'mobile_adas3d_outputs/parking_v0_dev'/RUN_ID/'portable_checkpoints'
LOCAL_CAPSULE = Path('/content/parking_v0_dev_inputs.zip')
INPUTS = Path('/content/parking_v0_verified_inputs')
OUTPUT = Path('/content')/('parking_v0_results_'+RUN_ID)
LOCAL = Path('/content')/('parking_v0_scratch_'+RUN_ID)
SOURCE = Path('/content/Urban_Parking_V0_Source')
WEIGHTS = Path('/content/parking_v0_weights')
DEPS = Path('/content')/('parking_v0_deps_'+RUN_ID)
PYTHON = sys.executable
OUTPUT.mkdir(parents=True, exist_ok=True)
def run(command, name, cwd=None, task_env=True):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'; log.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=''; env['YOLO_AUTOINSTALL']='false'
    if task_env: env['PYTHONPATH']=str(DEPS)+os.pathsep+str(MOBILE)
    print('+',shlex.join(command),'\nLOCAL log:',log,flush=True)
    tail=collections.deque(maxlen=30)
    with log.open('w') as stream:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in p.stdout:
                print(line,end='',flush=True);stream.write(line);stream.flush();tail.append(line.rstrip())
            code=p.wait()
        except BaseException:
            p.terminate();p.wait();raise
        finally:p.stdout.close()
        stream.write(f'\nExit code: {code}\n')
    if code:raise RuntimeError(f'Exit {code}; local files are preserved; inspect {log}\n'+'\n'.join(tail))
    return log
url='/'.join(['https:','','github.com','Ali-RT','mobile_adas3d.git'])
if not MOBILE.exists():run(['git','clone',url,MOBILE],'clone',task_env=False)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}; preserve other branches')
run(['git','pull','--ff-only'],'update_main',MOBILE,task_env=False)
SCRIPT=MOBILE/'scripts/parking_v0_local_io.py'
if not SCRIPT.is_file():raise FileNotFoundError('Local-first helper missing. Reload the latest main notebook.')
print(REVISION,'\nLocal output:',OUTPUT,'\nDrive checkpointing:',USE_DRIVE)

## 2 Prepare CPU dependencies and run tests
Install pinned preprocessing dependencies into a task-local folder. Use Colab's existing PyTorch on CPU. No notebook-kernel restart or old runtime receipt is needed. These tests use synthetic data and do not access your Drive dataset.

In [ ]:
run([PYTHON,'-m','pip','install','--quiet','--no-deps','--upgrade','--target',DEPS,
     'ultralytics==8.4.61','ultralytics-thop==2.0.18','numpy==2.1.3','pillow==11.3.0',
     'opencv-python-headless==5.0.0.93','scipy==1.16.3','ijson==3.4.0.post0'],'cpu_dependencies',MOBILE,task_env=False)
probe="import torch,cv2,numpy,PIL,scipy,ijson; from importlib.metadata import version; assert version('ultralytics')=='8.4.61'; assert cv2.__version__=='5.0.0'; print('CPU runtime:',torch.__version__,cv2.__version__,numpy.__version__,PIL.__version__)"
run([PYTHON,'-X','faulthandler','-c',probe],'cpu_preflight',MOBILE)
for pattern in ['test_parking_v0_local_io.py','test_parking_v0_dev.py','test_parking_v0_v1_benchmark.py']:
    run([PYTHON,'-m','unittest','discover','-s','tests','-p',pattern,'-v'],'tests_'+pattern,MOBILE)

## 3 Stage one reusable dev-only capsule
If the local capsule already exists, this cell verifies/reuses it without checking Drive inputs. Otherwise it copies the saved capsule once. Only the first-ever run packages the original 20 dev shards and checkpoint into a capsule and saves that capsule to Drive. SHA256, exact member inventory, model identity and frozen benchmark identity are checked before staging. No held-out image bodies are included.

An interrupted capsule transfer repeats that single transfer. Completed local input and prediction files are reused. Do not activate a GPU for staging. If a Drive quota is active and no local capsule is available, stop here until access recovers.

In [ ]:
command=[PYTHON,'-u',SCRIPT,'stage-inputs','--local-archive',LOCAL_CAPSULE,'--inputs',INPUTS]
if USE_DRIVE:
    command += ['--archive',REMOTE_CAPSULE,'--benchmark',BENCHMARK_SOURCE,'--bundle',MODEL_SOURCE]
elif not LOCAL_CAPSULE.is_file():
    raise FileNotFoundError(f'Upload the previously prepared capsule to {LOCAL_CAPSULE} in the Files panel, then rerun this cell. Do not upload the 45 GB dataset.')
run(command,'stage_local_inputs',MOBILE)
receipt=json.loads((INPUTS/'local_input_receipt.json').read_text())
assert receipt['dev_images']==512 and receipt['dev_shards']==20 and receipt['heldout_shard_bodies_included']==0
print('Verified local inputs ready. No Drive input reads are needed during inference.')
print('Keep this capsule for future runs:',LOCAL_CAPSULE)

## 4 Evaluate locally and checkpoint in batches
The original numerical evaluator is unchanged. Every image/GT read, prediction write and log write happens on local disk. With USE_DRIVE=True, the wrapper restores one archived checkpoint if needed, then saves one progress ZIP after every four completed dev data shards and one final ZIP. No individual prediction files are written to Drive. If a Drive save fails, evaluation continues locally and further Drive saves are disabled for this invocation; download the results before ending the VM.

The 512-image dev result does not authorize V1 training or held-out scoring. Model weights and BatchNorm remain unchanged.

In [ ]:
command=[PYTHON,'-X','faulthandler','-u',SCRIPT,'evaluate-local',
         '--inputs',INPUTS,'--output',OUTPUT,'--local',LOCAL,'--source',SOURCE,'--weights',WEIGHTS,
         '--checkpoint-every','4']
if USE_DRIVE:command += ['--checkpoints',REMOTE_CHECKPOINTS]
run(command,'v0_local_evaluation',MOBILE)
result=json.loads((OUTPUT/'v0_dev_results.json').read_text())
assert result['complete'] and result['metrics']['images']==512 and result['role']=='dev'
assert result['model_weights_and_bn_unchanged'] and result['optimizer_steps']==0
assert result['heldout_images_opened']==0 and not result['v1_trained']
print('Conditional primary four-cell MAE (m):',result['metrics']['conditional_equal_four_cell_mae_m'])
for name,cell in result['metrics']['cells'].items():
    print(name,'MAE:',cell['conditional_log_track_observation_macro_mae_m'],'coverage:',cell['prediction_coverage'],
          'tracks:',cell['reference_instance_tracks'],'logs:',cell['reference_recording_logs'])

## 5 Download the results before closing Colab
Send parking_v0_dev_results.zip for review. With Drive checkpointing, v0_complete.zip also preserves all predictions and registration for later resumption. If a Drive warning appeared, download that complete progress archive as well. It is under OUTPUT/portable_progress in the Files panel. Logs remain local; download them if troubleshooting is needed. No images are included in the result ZIP.

In [ ]:
RESULTS=OUTPUT/'parking_v0_dev_results.zip'
if not RESULTS.is_file():raise FileNotFoundError('Evaluation is incomplete. Rerun section 4 with the same RUN_ID and CPU runtime.')
print('Send this result:',RESULTS,'Bytes:',RESULTS.stat().st_size)
print('Portable complete progress:',OUTPUT/'portable_progress/v0_complete.zip')
from google.colab import files
files.download(str(RESULTS))